# Практикум: моделирование сложных систем

В ноутбуке представлены две модели: графовая система массового обслуживания и игра стратегий.

## 1. Подготовка окружения
Установите зависимости из `requirements.txt`, если они ещё не установлены.

In [ ]:
%pip install simpy networkx numpy pandas matplotlib

In [ ]:
from queue_model.simulation import QueueConfig, simulate_bank
from queue_model.graph import draw_bank_graph
from game_model.game import play_match
import pandas as pd
import matplotlib.pyplot as plt

## 2. Графовая модель банка
Система включает поступление клиентов, регистрацию, общую очередь, несколько окон и выход. Поступления и длительности обслуживания моделируются экспоненциальными распределениями.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
draw_bank_graph(4, ax)
plt.show()

In [ ]:
cfg = QueueConfig(horizon=480, arrival_rate=0.32, service_rate=0.12, clerks=4, patience=45, seed=42)
bank = simulate_bank(cfg)
{key: value for key, value in bank.items() if key not in ('records', 'queue', 'config')}

In [ ]:
queue_df = pd.DataFrame(bank['queue'], columns=['Время, мин', 'Длина очереди'])
queue_df.set_index('Время, мин').plot(figsize=(10, 4), title='Длина очереди во времени')
plt.ylabel('Клиенты')
plt.show()

## 3. Анализ числа окон
Сравним среднее ожидание при разных значениях числа специалистов. Для статистически более устойчивого вывода эксперимент следует повторить с несколькими начальными значениями генератора.

In [ ]:
rows = []
for clerks in [2, 3, 4, 5, 6]:
    r = simulate_bank(QueueConfig(clerks=clerks, seed=42))
    rows.append({'Окон': clerks, 'Обслужено': r['served'], 'Среднее ожидание, мин': r['mean_wait'], 'Отказались ждать': r['abandoned']})
bank_comparison = pd.DataFrame(rows)
bank_comparison

In [ ]:
bank_comparison.set_index('Окон')[['Среднее ожидание, мин']].plot(kind='bar', legend=False, figsize=(8, 4), title='Среднее ожидание в зависимости от числа окон')
plt.ylabel('Минуты')
plt.show()

## 4. Игровая модель противодействия
В игре «Камень, ножницы, бумага» агенты выбирают ходы одновременно. Адаптивный агент использует частоты ходов соперника на последних 20 раундах и выбирает действие, которое побеждает наиболее вероятный ход.

In [ ]:
game = play_match('Адаптивная', 'Циклическая', rounds=1000, seed=42)
{k: v for k, v in game.items() if k != 'history'}

In [ ]:
game_df = pd.DataFrame(game['history'])
game_df.set_index('Раунд')[['Счёт A', 'Счёт B']].plot(figsize=(10, 4), title='Динамика счёта')
plt.ylabel('Накопленный счёт')
plt.show()

## 5. Сравнение стратегий
Результат одной партии зависит от случайного начального значения и особенностей стратегий. Сравним адаптивного агента с несколькими соперниками.

In [ ]:
rows = []
for opponent in ['Случайная', 'Постоянная', 'Циклическая', 'Адаптивная']:
    r = play_match('Адаптивная', opponent, rounds=2000, seed=7)
    rows.append({'Стратегия B': opponent, 'Победы A': r['wins_a'], 'Победы B': r['wins_b'], 'Ничьи': r['draws'], 'Счёт A': r['score_a']})
game_comparison = pd.DataFrame(rows)
game_comparison

## 6. Выводы
В первой модели изменение числа окон и соотношения интенсивностей поступления и обслуживания влияет на ожидание и число клиентов, покидающих очередь. Во второй модели результат зависит от структуры поведения соперника: адаптация может быть полезна против предсказуемых стратегий, но не гарантирует преимущество против случайного или адаптивного игрока. Для окончательных выводов следует провести серию повторных запусков и сравнить средние показатели.